# SurviveCity v2 — Kaggle T4 training

Trains the v2 GRPO LoRA on a single T4 (~15 GB VRAM). Config is scaled
down from the DGX defaults so it actually fits:

| Knob | DGX (30GB) | Kaggle T4 (15GB) |
|---|---|---|
| Base precision | bf16 (`--no-4bit`) | **4-bit nf4** (forced — T4 too small for fp16) |
| `--num-generations` | 12 | **4** |
| `--max-completion-length` | 512 | **256** |
| `--lora-r` | 32 | **16** |
| `--lora-alpha` | 64 | **32** |
| `--max-steps` | 15 | 15 (same) |
| `--save-steps` | 1 | 1 (same — save every step) |

## Before you run

1. **Settings → Accelerator → T4 x2** (we'll bind to GPU 0 only — dual-T4
   + bitsandbytes triggers a DataParallel/quantize bug)
2. **Settings → Internet → On**
3. **Add-ons → Secrets → Add `HF_TOKEN`** with a write-scoped HuggingFace
   token (token never appears in chat or in the notebook)
4. **Settings → Persistence → Files only** so checkpoints survive session restarts

Total expected wallclock: **~2-3 hours** for 15 steps on T4 (matches v1's
Colab T4 run timing). Hub push happens after every step → all 15 checkpoints
land in `noanya/zombiee-v2` automatically.

## Cell 1 — Environment + dep installation

Bakes in every fix learned from the DGX Dockerfile saga:
- pinned modern stack (transformers 4.46.3, peft 0.13.2, trl 0.15.2)
- `mergekit` installed (trl 0.15+ chain-imports it)
- `llm_blender` STUBBED (real package has dep cascade hell — we don't use judges)
- `torchao` UNINSTALLED (peft check fires when both old torchao + modern peft installed)
- `TRANSFORMERS_CACHE` alias appended to `transformers/utils/hub.py`

Runs in ~5-7 minutes. Output is verbose; scroll to the smoke-test cell at
the end to confirm `FULL TRAINING IMPORT CHAIN OK`.

In [ ]:
import subprocess, sys, os

def pip_install(*pkgs, no_deps=False):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir"]
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(pkgs)
    print(f">>> pip install {'--no-deps ' if no_deps else ''}{' '.join(pkgs)}")
    subprocess.check_call(cmd)

# Pinned modern stack — same set as v2/Dockerfile.dgx
pip_install(
    "transformers==4.46.3",
    "accelerate==1.1.1",
    "peft==0.13.2",
    "datasets==3.1.0",
    "trl==0.15.2",
)

# 4-bit support (mandatory for T4 — base model in fp16 won't fit alongside training peak)
pip_install("bitsandbytes>=0.43")

# trl 0.15+ chain-imports mergekit unconditionally via callbacks.py
pip_install("mergekit")

# Plotting / logging / hub
pip_install("matplotlib>=3.8", "tensorboard", "huggingface_hub>=0.20", "wandb")

print(">>> Core packages installed.")

## Cell 2 — Apply the dep-cascade workarounds

1. Uninstall `torchao` (peft >= 0.13's strict version check fires when both
   old torchao + modern peft are installed; absent torchao → check returns
   False cleanly)
2. Patch `transformers/utils/hub.py` to add the legacy `TRANSFORMERS_CACHE`
   constant (some libs reference the old name even after rename)
3. Replace any installed `llm_blender` with a 6-line stub package — trl
   imports it unconditionally but our GRPO code never uses judges, so the
   stub is sufficient and avoids dragging in dataclasses_json,
   sentence-transformers, pytorch-lightning

In [ ]:
import subprocess, sys, os, glob

# 1) Uninstall torchao if present (best-effort; ignore failure)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
               check=False, capture_output=True)

# 2) Patch transformers/utils/hub.py — append TRANSFORMERS_CACHE alias if missing
import transformers.utils.hub as hub_mod
hub_path = hub_mod.__file__
with open(hub_path, "r") as f:
    src = f.read()
if "TRANSFORMERS_CACHE" not in src:
    with open(hub_path, "a") as f:
        f.write(
            "\n# Legacy alias for libs that haven't caught up to the rename (added by Kaggle notebook)\n"
            'TRANSFORMERS_CACHE = HF_HUB_CACHE if "HF_HUB_CACHE" in dir() else "~/.cache/huggingface/hub"\n'
        )
    print(f">>> Patched {hub_path} with TRANSFORMERS_CACHE alias")
else:
    print(f">>> {hub_path} already has TRANSFORMERS_CACHE")

# 3) Stub out llm_blender — replace any pypi install with our minimal class
site_packages_dirs = glob.glob("/usr/lib/python3*/dist-packages") + \
                     glob.glob("/usr/lib/python3*/site-packages") + \
                     glob.glob("/opt/conda/lib/python3*/site-packages") + \
                     glob.glob("/usr/local/lib/python3*/dist-packages")

import sysconfig
site_packages = sysconfig.get_paths()["purelib"]
print(f">>> site-packages: {site_packages}")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "llm-blender"],
               check=False, capture_output=True)

stub_dir = os.path.join(site_packages, "llm_blender")
if os.path.exists(stub_dir):
    import shutil
    shutil.rmtree(stub_dir)
os.makedirs(stub_dir, exist_ok=True)

stub_code = '''"""Stub llm_blender for v2 Kaggle notebook (real package has dep conflicts)."""

class Blender:
    """Stub. Real llm_blender is not installed."""
    def __init__(self, *args, **kwargs):
        raise RuntimeError(
            "llm_blender.Blender is stubbed. SurviveCity v2 GRPO doesn't use judges."
        )
    def loadranker(self, *args, **kwargs): pass
    def loadfuser(self, *args, **kwargs): pass
    def rank(self, *args, **kwargs): raise RuntimeError("llm_blender stubbed")
    def fuse(self, *args, **kwargs): raise RuntimeError("llm_blender stubbed")
'''
with open(os.path.join(stub_dir, "__init__.py"), "w") as f:
    f.write(stub_code)
print(f">>> Wrote llm_blender stub to {stub_dir}/__init__.py")

# Force re-import in case anything was already cached
for mod in list(sys.modules):
    if mod == "llm_blender" or mod.startswith("llm_blender."):
        del sys.modules[mod]

print(">>> Workarounds applied.")

## Cell 3 — Smoke test the import chain

Mirror of the smoke test inside `v2/Dockerfile.dgx`. Exercises every module
the training script imports. If this cell prints `FULL TRAINING IMPORT
CHAIN OK` you're guaranteed the runtime imports won't fail.

If this cell errors, **STOP** and read the traceback — re-running training
won't fix it. Most likely cause: cell 2's stub creation didn't land in the
right `site-packages` directory; check the `>>> site-packages:` line above.

In [ ]:
import importlib.util
import torch
import transformers, peft, datasets, accelerate, bitsandbytes, huggingface_hub, trl
import mergekit, llm_blender
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.utils.hub import TRANSFORMERS_CACHE
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training, PeftModel
from trl import GRPOTrainer, GRPOConfig

ta = importlib.util.find_spec("torchao")
print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"peft         {peft.__version__}")
print(f"datasets     {datasets.__version__}")
print(f"accelerate   {accelerate.__version__}")
print(f"trl          {trl.__version__}")
print(f"bitsandbytes {bitsandbytes.__version__}")
print(f"mergekit     present")
print(f"llm_blender  stub (TRANSFORMERS_CACHE alias = {TRANSFORMERS_CACHE})")
print(f"torchao      {'absent (intended)' if ta is None else 'PRESENT (unexpected)'}")
assert ta is None, "torchao should be uninstalled"

# GPU sanity
if torch.cuda.is_available():
    free_gb = torch.cuda.mem_get_info(0)[0] / 1e9
    total_gb = torch.cuda.mem_get_info(0)[1] / 1e9
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU 0       {torch.cuda.get_device_name(0)} cc={cap[0]}.{cap[1]} "
          f"VRAM free={free_gb:.1f}GB total={total_gb:.1f}GB")
    if total_gb < 14:
        print("⚠ GPU < 14 GB — config may need further scaling")
else:
    print("⚠ NO GPU detected")

print("\nFULL TRAINING IMPORT CHAIN OK")

## Cell 4 — Pull the v2 source tree from GitHub

The repo is public (or private with token-gated access — both work).
We `cd` into `v2/` so subsequent `python -m training.train` works.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/SirjanSingh/zombiee.git"
WORK_DIR = "/kaggle/working/zombiee"
V2_DIR = os.path.join(WORK_DIR, "v2")

if os.path.exists(WORK_DIR):
    print(">>> Repo already cloned, pulling latest")
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)
else:
    print(">>> Cloning fresh")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK_DIR], check=True)

os.chdir(V2_DIR)
print(f">>> cwd: {os.getcwd()}")
print(f">>> v2 contents:")
for entry in sorted(os.listdir(".")):
    print("    ", entry)

## Cell 5 — Read HF token from Kaggle Secrets

**Token NEVER appears in chat or in the notebook source.** Add `HF_TOKEN`
via Add-ons → Secrets in the Kaggle UI, then this cell reads it via the
official Kaggle SDK.

Also tests auth before training so you don't burn 2 hours on a bad token.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
try:
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Could not read 'HF_TOKEN' from Kaggle Secrets. "
        "Go to Add-ons → Secrets, add HF_TOKEN with your write-scoped token, "
        "then re-run this cell."
    ) from e

# Inject for both env-var conventions (older transformers reads HUGGINGFACE_TOKEN,
# newer huggingface_hub reads HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN

# Auth pre-flight
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
user_info = api.whoami()
print(f">>> Authed as: {user_info.get('name', '?')}  type={user_info.get('type', '?')}")

# Verify the target repo exists / is writable
HUB_REPO = "noanya/zombiee-v2"
try:
    info = api.repo_info(HUB_REPO)
    print(f">>> Repo {HUB_REPO} exists. private={info.private} sha={info.sha[:8] if info.sha else 'n/a'}")
except Exception as e:
    print(f">>> Repo {HUB_REPO} does not exist yet — will be created on first push.")

# Optional: tiny upload to confirm write permission BEFORE training
import tempfile
with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
    f.write("kaggle-auth-test\n")
    test_path = f.name
try:
    api.upload_file(
        path_or_fileobj=test_path,
        path_in_repo=".kaggle_auth_test.txt",
        repo_id=HUB_REPO,
        commit_message="kaggle auth test (will be deleted)",
    )
    api.delete_file(
        path_in_repo=".kaggle_auth_test.txt",
        repo_id=HUB_REPO,
        commit_message="cleanup auth test",
    )
    print(f">>> WRITE access to {HUB_REPO} confirmed")
except Exception as e:
    print(f"⚠ WRITE test FAILED: {e}")
    print("  Token likely has read-only scope. Generate a write token at huggingface.co/settings/tokens")
    raise
finally:
    os.remove(test_path)

## Cell 6 — Kaggle-specific GPU config

**Critical for Kaggle's dual-T4 setup**: bitsandbytes' 4-bit path can't
co-exist with PyTorch's auto DataParallel when two GPUs are visible. v1
hit this hard. Fix is to expose only GPU 0 to the training process.

Also enables PyTorch's expandable-segments allocator to reduce fragmentation
during long GRPO runs.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# TF32 / bf16 detection — V100/T4 can't do bf16 natively
import torch
print(f"GPU(s) visible:        {torch.cuda.device_count()}")
print(f"GPU 0:                 {torch.cuda.get_device_name(0)}")
cap = torch.cuda.get_device_capability(0)
print(f"Compute capability:    {cap[0]}.{cap[1]}")
print(f"bf16 supported:        {torch.cuda.is_bf16_supported()}")
print(f"VRAM total (GB):       {torch.cuda.mem_get_info(0)[1]/1e9:.2f}")
print(f"CUDA_VISIBLE_DEVICES:  {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"ALLOC_CONF:            {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")

## Cell 7 — Launch training

Calls the same `training.train` entrypoint the Docker container uses, with
T4-sized config:

- `--no-4bit` is **NOT** set → uses bnb 4-bit nf4 (mandatory on T4)
- `--num-generations 4` (DGX uses 12)
- `--max-completion-length 256` (DGX uses 512)
- `--lora-r 16 --lora-alpha 32` (DGX uses 32/64)
- `--vram-reserve-gb 0` (single-tenant on Kaggle, no holder needed)
- `--max-steps 15 --save-steps 1` → 15 saves, every step pushed to Hub

Output streams live. Per-step time on T4: ~10-15 minutes. Total: ~3 hours.

In [ ]:
import subprocess, sys, os

# Re-affirm cwd in case the notebook kernel reset
os.chdir("/kaggle/working/zombiee/v2")

cmd = [
    sys.executable, "-m", "training.train",
    "--model-name", "Qwen/Qwen2.5-3B-Instruct",
    "--max-steps", "15",
    "--save-steps", "1",
    "--save-total-limit", "15",
    "--num-generations", "4",
    "--grad-accum-steps", "12",
    "--max-completion-length", "256",
    "--lora-r", "16",
    "--lora-alpha", "32",
    "--max-seq-length", "4096",
    "--max-prompt-length", "1024",
    "--vram-reserve-gb", "0",
    "--push-to-hub",
    "--hub-model-id", "noanya/zombiee-v2",
    "--resume-from-checkpoint", "auto",
    "--report-to", "tensorboard",
    "--output-dir", "/kaggle/working/checkpoints",
]

print(">>> Launching training:")
print("   ", " ".join(cmd))
print()

# stream stdout/stderr live so we can watch progress
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()

print(f"\n>>> Training exited with code {proc.returncode}")
if proc.returncode != 0:
    raise RuntimeError(f"Training failed (exit {proc.returncode})")

## Cell 8 — Verify checkpoints landed on Hub

After training completes, this cell lists the files on `noanya/zombiee-v2`
to confirm all 15 checkpoints pushed successfully.

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi(token=os.environ["HF_TOKEN"])
files = api.list_repo_files("noanya/zombiee-v2")

print(f">>> Files on noanya/zombiee-v2 ({len(files)} total):")
for f in sorted(files):
    print("   ", f)

checkpoints = sorted({f.split('/')[0] for f in files if f.startswith('checkpoint-')})
print(f"\n>>> Checkpoints found: {len(checkpoints)} → {checkpoints}")
if len(checkpoints) >= 15:
    print(">>> All 15 checkpoints pushed successfully.")
elif len(checkpoints) > 0:
    print(f">>> Only {len(checkpoints)} checkpoints — training may have stopped early.")
else:
    print(">>> NO checkpoint subdirs. Either training is still running or pushes failed.")

## Cell 9 — Quick TensorBoard view (optional)

If the training cell finished cleanly, the run's TB events live in
`/kaggle/working/checkpoints/runs/`. Render them inline.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/checkpoints/runs

## Done

All 15 LoRA checkpoints are now on `noanya/zombiee-v2`. To evaluate any of
them on a separate 15GB box, run:

```bash
python -m training.eval --lora-path noanya/zombiee-v2 --eval-step 15
```

The eval pulls the latest adapter from Hub and runs the 9-metric evaluation
(survival rate, mean reward, episode length, vote phase accuracy, medication
ROI, infection containment, etc.). See `v2/training/eval.py` and
`.planning2/07_V2_DESIGN_AND_IMPL.md` for details.